In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [11]:
df = pd.read_excel("Clientes datos incompletos.xlsx")
df.head()

,ID_CLIENTE,CELULAR,EMAIL,NOMBRE,APELLIDO
0,1183191,305 805342,claudia.scoss@gmail.com,CLAUDIA DE SOUZA,ARAUJO
1,1770193,1111111111,GARCIAJESUS@GMAIL.COM,JESUS ALEJANDRO,GARCIA
2,5583530,68398634,GABRIELJDJ@GMAIL.COM,JESUS,GABRIEL
3,7464039,1111111111,manuelemercadop@gmail.com,MANUEL,MERCADO
4,10058256,1111111111,fmejiag@gmail.com,FERNANDO,MEJIA GUTIERRES


In [3]:
# Normalizar columnas si tienen espacios o nombres extraños
df.columns = df.columns.str.strip().str.lower()

In [12]:
print(df.columns.tolist())

['ID_CLIENTE', 'CELULAR', 'EMAIL', 'NOMBRE', 'APELLIDO']


In [ ]:

# Eliminar registros sin correo ni apellido
df['NOMBRE'] = df['NOMBRE'].replace("", pd.NA)
df = df.dropna(subset=['ID_CLIENTE', 'CELULAR', 'EMAIL', 'NOMBRE', 'APELLIDO'])

In [18]:
# Limpiar columna de celular
df['CELULAR'] = df['CELULAR'].astype(str)

# Eliminar espacios, paréntesis, guiones, etc.
df['CELULAR'] = df['CELULAR'].str.replace(r'[^\d+]', '', regex=True)

# Eliminar prefijo +57 si existe
df['CELULAR'] = df['CELULAR'].str.replace(r'^\+?57', '', regex=True)

# Asegurarse que queda solo con números
df['CELULAR'] = df['CELULAR'].str.replace(r'\D', '', regex=True)

# Filtrar: exactamente 10 dígitos y que empiece con 3
df = df[df['CELULAR'].str.match(r'^3\d{9}$')]

# Eliminar celulares claramente inválidos
celulares_invalidos = {
    '1111111111', '1234567890', '0000000000', '9999999999', '2222222222'
}
df = df[~df['CELULAR'].isin(celulares_invalidos)]

In [21]:

# Validar que el correo tenga una estructura básica válida
#df = df[df['EMAIL'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True)]

df = df[df['EMAIL'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True, na=False)]


In [22]:
# Resetear el índice
df = df.reset_index(drop=True)

In [24]:
# Guardar archivo limpio
df.to_excel('base_limpia.xlsx', index=False)